<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Excel avanzado: estilos, gráficos e informes profesionales**

## *De workbook funcional a informe listo para entregar*

En el laboratorio anterior aprendimos a crear y manipular Excel con `openpyxl`.

Ahora daremos el siguiente paso:

> **automatizar la presentación, el reporting y la generación de informes visuales**

Trabajaremos con:

- estilos reutilizables;
- formatos condicionales;
- tablas de Excel;
- gráficos;
- hojas de dashboard;
- KPIs;
- resúmenes ejecutivos;
- validaciones;
- plantillas;
- generación automática de informes.

## Objetivos

Al finalizar deberías poder:

1. Aplicar estilos consistentes.
2. Crear funciones reutilizables de formato.
3. Crear tablas de Excel.
4. Aplicar formato condicional.
5. Crear gráficos de barras, líneas y circular.
6. Configurar títulos, leyendas y ejes.
7. Insertar gráficos en hojas.
8. Crear una hoja de dashboard.
9. Crear KPIs destacados.
10. Crear resúmenes automáticos.
11. Generar informes con múltiples hojas.
12. Reutilizar una plantilla.
13. Validar que el informe final contiene todos los elementos esperados.
14. Automatizar un informe completo de principio a fin.

## 1. Preparación

In [ ]:
from pathlib import Path
import shutil
import random

import pandas as pd
import matplotlib.pyplot as plt

try:
    import openpyxl

    from openpyxl import Workbook, load_workbook

    from openpyxl.styles import (
        Font,
        PatternFill,
        Alignment,
        Border,
        Side
    )

    from openpyxl.chart import (
        BarChart,
        LineChart,
        PieChart,
        Reference
    )

    from openpyxl.formatting.rule import (
        CellIsRule,
        ColorScaleRule
    )

    from openpyxl.worksheet.table import (
        Table,
        TableStyleInfo
    )

    from openpyxl.utils import (
        get_column_letter
    )

    OPENPYXL_DISPONIBLE = True

except ImportError:
    OPENPYXL_DISPONIBLE = False

print(
    "openpyxl disponible:",
    OPENPYXL_DISPONIBLE
)

if OPENPYXL_DISPONIBLE:
    print(
        "Versión:",
        openpyxl.__version__
    )

## 2. Crear entorno

In [ ]:
raiz = Path(
    "demo_excel_informes"
)

if raiz.exists():
    shutil.rmtree(
        raiz
    )

raiz.mkdir()

ruta_excel = (
    raiz
    / "informe_ventas.xlsx"
)

print(
    ruta_excel.resolve()
)

## 3. Crear dataset más realista

In [ ]:
random.seed(
    42
)

regiones = [
    "Norte",
    "Sur",
    "Este",
    "Oeste"
]

productos = [
    "Producto A",
    "Producto B",
    "Producto C",
    "Producto D"
]

filas = []

for i in range(
    1,
    201
):
    region = random.choice(
        regiones
    )

    producto = random.choice(
        productos
    )

    unidades = random.randint(
        1,
        20
    )

    precio = random.choice([
        25,
        40,
        60,
        100
    ])

    descuento = random.choice([
        0,
        0.05,
        0.10,
        0.15
    ])

    filas.append({
        "ID":
            i,

        "Region":
            region,

        "Producto":
            producto,

        "Unidades":
            unidades,

        "Precio":
            precio,

        "Descuento":
            descuento,

        "Importe":
            unidades
            * precio
            * (
                1
                - descuento
            )
    })

df = pd.DataFrame(
    filas
)

df.head()

## 3.1 Resumen descriptivo

In [ ]:
df.describe(
    include="all"
)

## 3.2 Visualización previa

In [ ]:
resumen_region = (
    df.groupby(
        "Region",
        as_index=False
    )
    .agg(
        Facturacion=(
            "Importe",
            "sum"
        ),
        Unidades=(
            "Unidades",
            "sum"
        )
    )
)

resumen_region

In [ ]:
plt.figure(
    figsize=(8, 5)
)

plt.bar(
    resumen_region[
        "Region"
    ],
    resumen_region[
        "Facturacion"
    ]
)

plt.ylabel(
    "Facturación"
)

plt.title(
    "Facturación por región"
)

plt.tight_layout()
plt.show()

### Interpretación

Antes de construir el dashboard conviene entender qué información queremos destacar.

En este caso, la facturación por región será uno de los principales indicadores visuales.

## 4. Crear workbook base

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb = Workbook()

    ws_datos = wb.active
    ws_datos.title = "Datos"

    ws_resumen = wb.create_sheet(
        "Resumen"
    )

    ws_dashboard = wb.create_sheet(
        "Dashboard"
    )

    print(
        wb.sheetnames
    )

## 5. Exportar DataFrame a hoja

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_datos.append(
        list(
            df.columns
        )
    )

    for fila in df.itertuples(
        index=False,
        name=None
    ):
        ws_datos.append(
            list(
                fila
            )
        )

    print(
        "Filas:",
        ws_datos.max_row
    )

## 6. Crear funciones de estilo reutilizable

In [ ]:
if OPENPYXL_DISPONIBLE:

    COLOR_CABECERA = "1F4E78"
    COLOR_KPI = "D9EAF7"
    COLOR_ALERTA = "FCE4D6"
    COLOR_OK = "E2F0D9"

    borde_fino = Side(
        style="thin",
        color="BFBFBF"
    )

    borde = Border(
        left=borde_fino,
        right=borde_fino,
        top=borde_fino,
        bottom=borde_fino
    )


    def aplicar_cabecera(
        worksheet,
        fila=1,
        color=COLOR_CABECERA
    ):
        for celda in worksheet[
            fila
        ]:
            celda.font = Font(
                bold=True,
                color="FFFFFF"
            )

            celda.fill = PatternFill(
                fill_type="solid",
                fgColor=color
            )

            celda.alignment = Alignment(
                horizontal="center"
            )

            celda.border = borde


    def ajustar_anchos(
        worksheet,
        max_width=30
    ):
        for columna in worksheet.columns:

            letra = get_column_letter(
                columna[
                    0
                ].column
            )

            longitud = max(
                len(
                    str(
                        celda.value
                    )
                )
                if celda.value
                is not None
                else 0
                for celda
                in columna
            )

            worksheet.column_dimensions[
                letra
            ].width = min(
                longitud + 2,
                max_width
            )


    def aplicar_bordes(
        worksheet,
        min_row=1
    ):
        for fila in worksheet.iter_rows(
            min_row=min_row,
            max_row=worksheet.max_row,
            min_col=1,
            max_col=worksheet.max_column
        ):
            for celda in fila:
                celda.border = borde

    print(
        "Funciones de estilo preparadas."
    )

## 7. Aplicar estilo a Datos

In [ ]:
if OPENPYXL_DISPONIBLE:

    aplicar_cabecera(
        ws_datos
    )

    aplicar_bordes(
        ws_datos
    )

    ajustar_anchos(
        ws_datos
    )

    ws_datos.freeze_panes = "A2"

    for fila in range(
        2,
        ws_datos.max_row + 1
    ):

        ws_datos.cell(
            fila,
            5
        ).number_format = '#,##0.00 €'

        ws_datos.cell(
            fila,
            6
        ).number_format = '0%'

        ws_datos.cell(
            fila,
            7
        ).number_format = '#,##0.00 €'

    print(
        "Hoja Datos formateada."
    )

## 8. Convertir rango en tabla Excel

In [ ]:
if OPENPYXL_DISPONIBLE:

    referencia = (
        f"A1:G{ws_datos.max_row}"
    )

    tabla = Table(
        displayName="TablaVentas",
        ref=referencia
    )

    estilo = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False
    )

    tabla.tableStyleInfo = estilo

    ws_datos.add_table(
        tabla
    )

    print(
        "Tabla creada:",
        referencia
    )

### Interpretación

Una tabla de Excel mejora:

- lectura;
- filtros;
- estilo;
- navegación.

Además facilita trabajar con informes más próximos al uso real de Excel.

## 9. Formato condicional

Vamos a destacar:

- facturación baja;
- facturación alta;
- distribución relativa de importes.

## 9.1 Regla de valor alto

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_datos.conditional_formatting.add(
        f"G2:G{ws_datos.max_row}",
        CellIsRule(
            operator="greaterThan",
            formula=[
                "1000"
            ],
            fill=PatternFill(
                fill_type="solid",
                fgColor="C6EFCE"
            )
        )
    )

    print(
        "Formato condicional aplicado."
    )

## 9.2 Escala de color

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_datos.conditional_formatting.add(
        f"G2:G{ws_datos.max_row}",
        ColorScaleRule(
            start_type="min",
            start_color="F8696B",
            mid_type="percentile",
            mid_value=50,
            mid_color="FFEB84",
            end_type="max",
            end_color="63BE7B"
        )
    )

### Interpretación

El formato condicional permite transformar números en señales visuales sin modificar los datos.

## 10. Crear resumen por región

In [ ]:
resumen_region = (
    df.groupby(
        "Region",
        as_index=False
    )
    .agg(
        Facturacion=(
            "Importe",
            "sum"
        ),
        Operaciones=(
            "ID",
            "count"
        ),
        Unidades=(
            "Unidades",
            "sum"
        ),
        Ticket_Medio=(
            "Importe",
            "mean"
        )
    )
)

resumen_region

## 10.1 Escribir Resumen

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_resumen.append(
        list(
            resumen_region.columns
        )
    )

    for fila in resumen_region.itertuples(
        index=False,
        name=None
    ):
        ws_resumen.append(
            list(
                fila
            )
        )

    aplicar_cabecera(
        ws_resumen
    )

    aplicar_bordes(
        ws_resumen
    )

    ajustar_anchos(
        ws_resumen
    )

    for fila in range(
        2,
        ws_resumen.max_row + 1
    ):
        ws_resumen.cell(
            fila,
            2
        ).number_format = '#,##0.00 €'

        ws_resumen.cell(
            fila,
            5
        ).number_format = '#,##0.00 €'

    print(
        "Resumen creado."
    )

## 11. Crear gráfico de barras

In [ ]:
if OPENPYXL_DISPONIBLE:

    grafico_barras = BarChart()

    grafico_barras.title = (
        "Facturación por región"
    )

    grafico_barras.y_axis.title = (
        "Facturación"
    )

    grafico_barras.x_axis.title = (
        "Región"
    )

    datos = Reference(
        ws_resumen,
        min_col=2,
        min_row=1,
        max_row=ws_resumen.max_row
    )

    categorias = Reference(
        ws_resumen,
        min_col=1,
        min_row=2,
        max_row=ws_resumen.max_row
    )

    grafico_barras.add_data(
        datos,
        titles_from_data=True
    )

    grafico_barras.set_categories(
        categorias
    )

    grafico_barras.height = 8
    grafico_barras.width = 14

    ws_resumen.add_chart(
        grafico_barras,
        "G2"
    )

    print(
        "Gráfico de barras añadido."
    )

## 12. Crear gráfico circular

In [ ]:
if OPENPYXL_DISPONIBLE:

    grafico_pie = PieChart()

    grafico_pie.title = (
        "Peso de facturación por región"
    )

    datos = Reference(
        ws_resumen,
        min_col=2,
        min_row=1,
        max_row=ws_resumen.max_row
    )

    categorias = Reference(
        ws_resumen,
        min_col=1,
        min_row=2,
        max_row=ws_resumen.max_row
    )

    grafico_pie.add_data(
        datos,
        titles_from_data=True
    )

    grafico_pie.set_categories(
        categorias
    )

    grafico_pie.height = 8
    grafico_pie.width = 10

    ws_resumen.add_chart(
        grafico_pie,
        "G18"
    )

    print(
        "Gráfico circular añadido."
    )

## 13. Crear serie temporal simulada

In [ ]:
df_tiempo = pd.DataFrame({
    "Mes": [
        "Ene",
        "Feb",
        "Mar",
        "Abr",
        "May",
        "Jun"
    ],
    "Facturacion": [
        12000,
        13500,
        12900,
        15100,
        16500,
        17800
    ]
})

df_tiempo

## 13.1 Escribir serie temporal

In [ ]:
if OPENPYXL_DISPONIBLE:

    fila_inicio = (
        ws_resumen.max_row
        + 3
    )

    ws_resumen.cell(
        fila_inicio,
        1,
        "Mes"
    )

    ws_resumen.cell(
        fila_inicio,
        2,
        "Facturacion_Mensual"
    )

    for offset, fila in enumerate(
        df_tiempo.itertuples(
            index=False,
            name=None
        ),
        start=1
    ):
        ws_resumen.cell(
            fila_inicio + offset,
            1,
            fila[0]
        )

        ws_resumen.cell(
            fila_inicio + offset,
            2,
            fila[1]
        )

    print(
        "Serie temporal añadida."
    )

## 14. Crear gráfico de líneas

In [ ]:
if OPENPYXL_DISPONIBLE:

    grafico_lineas = LineChart()

    grafico_lineas.title = (
        "Evolución mensual"
    )

    grafico_lineas.y_axis.title = (
        "Facturación"
    )

    grafico_lineas.x_axis.title = (
        "Mes"
    )

    datos = Reference(
        ws_resumen,
        min_col=2,
        min_row=fila_inicio,
        max_row=fila_inicio
        + len(
            df_tiempo
        )
    )

    categorias = Reference(
        ws_resumen,
        min_col=1,
        min_row=fila_inicio + 1,
        max_row=fila_inicio
        + len(
            df_tiempo
        )
    )

    grafico_lineas.add_data(
        datos,
        titles_from_data=True
    )

    grafico_lineas.set_categories(
        categorias
    )

    grafico_lineas.height = 8
    grafico_lineas.width = 14

    ws_resumen.add_chart(
        grafico_lineas,
        "G34"
    )

    print(
        "Gráfico de líneas añadido."
    )

## 15. Diseñar dashboard

La hoja `Dashboard` debe responder rápidamente a preguntas de negocio.

Incluiremos:

- facturación total;
- operaciones;
- clientes/regiones;
- ticket medio;
- región líder;
- gráficos.

## 15.1 Calcular KPIs

In [ ]:
kpis = {
    "Facturación total":
        float(
            df[
                "Importe"
            ].sum()
        ),

    "Operaciones":
        int(
            len(
                df
            )
        ),

    "Unidades":
        int(
            df[
                "Unidades"
            ].sum()
        ),

    "Ticket medio":
        float(
            df[
                "Importe"
            ].mean()
        ),

    "Región líder":
        (
            resumen_region
            .sort_values(
                "Facturacion",
                ascending=False
            )
            .iloc[0][
                "Region"
            ]
        )
}

kpis

## 15.2 Escribir título

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_dashboard.merge_cells(
        "A1:H2"
    )

    celda_titulo = (
        ws_dashboard[
            "A1"
        ]
    )

    celda_titulo.value = (
        "Dashboard de Ventas"
    )

    celda_titulo.font = Font(
        bold=True,
        size=20,
        color="FFFFFF"
    )

    celda_titulo.fill = PatternFill(
        fill_type="solid",
        fgColor=COLOR_CABECERA
    )

    celda_titulo.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )

## 15.3 Crear tarjetas KPI

In [ ]:
if OPENPYXL_DISPONIBLE:

    posiciones = [
        ("A4", "B4"),
        ("D4", "E4"),
        ("G4", "H4"),
        ("A7", "B7"),
        ("D7", "E7")
    ]

    for (
        nombre,
        valor
    ), (
        celda_nombre,
        celda_valor
    ) in zip(
        kpis.items(),
        posiciones
    ):

        ws_dashboard[
            celda_nombre
        ] = nombre

        ws_dashboard[
            celda_valor
        ] = valor

        ws_dashboard[
            celda_nombre
        ].font = Font(
            bold=True
        )

        ws_dashboard[
            celda_nombre
        ].fill = PatternFill(
            fill_type="solid",
            fgColor=COLOR_KPI
        )

        ws_dashboard[
            celda_valor
        ].font = Font(
            bold=True,
            size=14
        )

        ws_dashboard[
            celda_nombre
        ].alignment = Alignment(
            horizontal="center"
        )

        ws_dashboard[
            celda_valor
        ].alignment = Alignment(
            horizontal="center"
        )

        ws_dashboard[
            celda_nombre
        ].border = borde

        ws_dashboard[
            celda_valor
        ].border = borde

    ws_dashboard[
        "B4"
    ].number_format = '#,##0.00 €'

    ws_dashboard[
        "B7"
    ].number_format = '#,##0.00 €'

    print(
        "KPIs creados."
    )

## 16. Copiar resumen al dashboard

In [ ]:
if OPENPYXL_DISPONIBLE:

    inicio_tabla = 11

    ws_dashboard.cell(
        inicio_tabla,
        1,
        "Region"
    )

    ws_dashboard.cell(
        inicio_tabla,
        2,
        "Facturacion"
    )

    ws_dashboard.cell(
        inicio_tabla,
        3,
        "Operaciones"
    )

    for offset, fila in enumerate(
        resumen_region.itertuples(
            index=False
        ),
        start=1
    ):
        ws_dashboard.cell(
            inicio_tabla + offset,
            1,
            fila.Region
        )

        ws_dashboard.cell(
            inicio_tabla + offset,
            2,
            fila.Facturacion
        )

        ws_dashboard.cell(
            inicio_tabla + offset,
            3,
            fila.Operaciones
        )

    aplicar_cabecera(
        ws_dashboard,
        fila=inicio_tabla
    )

    print(
        "Tabla resumen copiada."
    )

## 17. Crear gráfico en Dashboard

In [ ]:
if OPENPYXL_DISPONIBLE:

    grafico_dashboard = BarChart()

    grafico_dashboard.title = (
        "Facturación por región"
    )

    datos = Reference(
        ws_dashboard,
        min_col=2,
        min_row=inicio_tabla,
        max_row=inicio_tabla
        + len(
            resumen_region
        )
    )

    categorias = Reference(
        ws_dashboard,
        min_col=1,
        min_row=inicio_tabla + 1,
        max_row=inicio_tabla
        + len(
            resumen_region
        )
    )

    grafico_dashboard.add_data(
        datos,
        titles_from_data=True
    )

    grafico_dashboard.set_categories(
        categorias
    )

    grafico_dashboard.height = 9
    grafico_dashboard.width = 15

    ws_dashboard.add_chart(
        grafico_dashboard,
        "E11"
    )

    print(
        "Gráfico insertado en Dashboard."
    )

## 18. Ajustar columnas del dashboard

In [ ]:
if OPENPYXL_DISPONIBLE:

    for columna in range(
        1,
        9
    ):
        ws_dashboard.column_dimensions[
            get_column_letter(
                columna
            )
        ].width = 18

## 19. Ocultar líneas de cuadrícula

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_dashboard.sheet_view.showGridLines = False

    ws_resumen.sheet_view.showGridLines = False

    print(
        "Gridlines ocultas."
    )

## 20. Detectar una inconsistencia visual

Supongamos que una hoja tiene columnas excesivamente anchas.

Primero la provocamos.

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_dashboard.column_dimensions[
        "A"
    ].width = 80

    print(
        "Ancho A:",
        ws_dashboard.column_dimensions[
            "A"
        ].width
    )

## 20.1 Corregir

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_dashboard.column_dimensions[
        "A"
    ].width = 18

    print(
        "Ancho corregido:",
        ws_dashboard.column_dimensions[
            "A"
        ].width
    )

## 21. Crear una hoja de metadatos

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_meta = wb.create_sheet(
        "Metadatos"
    )

    metadatos = {
        "Informe":
            "Ventas",

        "Filas":
            len(
                df
            ),

        "Regiones":
            df[
                "Region"
            ].nunique(),

        "Productos":
            df[
                "Producto"
            ].nunique(),

        "Generado_por":
            "Python + openpyxl"
    }

    ws_meta.append([
        "Campo",
        "Valor"
    ])

    for clave, valor in metadatos.items():
        ws_meta.append([
            clave,
            valor
        ])

    aplicar_cabecera(
        ws_meta
    )

    ajustar_anchos(
        ws_meta
    )

### Interpretación

Una hoja de metadatos mejora la trazabilidad del informe y puede guardar:

- fuente;
- fecha;
- versión;
- número de registros;
- parámetros.

## 22. Guardar informe

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb.save(
        ruta_excel
    )

    print(
        "Informe guardado:",
        ruta_excel.exists()
    )

## 23. Reabrir y validar

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb_final = load_workbook(
        ruta_excel
    )

    print(
        wb_final.sheetnames
    )

## 23.1 Validar hojas esperadas

In [ ]:
if OPENPYXL_DISPONIBLE:

    esperadas = {
        "Datos",
        "Resumen",
        "Dashboard",
        "Metadatos"
    }

    assert esperadas.issubset(
        set(
            wb_final.sheetnames
        )
    )

    print(
        "Hojas validadas."
    )

## 23.2 Validar tabla Excel

In [ ]:
if OPENPYXL_DISPONIBLE:

    ws_validacion = wb_final[
        "Datos"
    ]

    print(
        "Tablas:",
        list(
            ws_validacion.tables.keys()
        )
    )

    assert (
        "TablaVentas"
        in ws_validacion.tables
    )

## 23.3 Validar gráficos

In [ ]:
if OPENPYXL_DISPONIBLE:

    print(
        "Gráficos en Resumen:",
        len(
            wb_final[
                "Resumen"
            ]._charts
        )
    )

    print(
        "Gráficos en Dashboard:",
        len(
            wb_final[
                "Dashboard"
            ]._charts
        )
    )

    assert (
        len(
            wb_final[
                "Resumen"
            ]._charts
        )
        >= 3
    )

    assert (
        len(
            wb_final[
                "Dashboard"
            ]._charts
        )
        >= 1
    )

## 24. Crear función reusable de KPI

In [ ]:
if OPENPYXL_DISPONIBLE:

    def escribir_kpi(
        worksheet,
        celda_nombre,
        celda_valor,
        nombre,
        valor,
        color=COLOR_KPI
    ):
        worksheet[
            celda_nombre
        ] = nombre

        worksheet[
            celda_valor
        ] = valor

        worksheet[
            celda_nombre
        ].font = Font(
            bold=True
        )

        worksheet[
            celda_nombre
        ].fill = PatternFill(
            fill_type="solid",
            fgColor=color
        )

        worksheet[
            celda_valor
        ].font = Font(
            bold=True,
            size=14
        )

        worksheet[
            celda_nombre
        ].alignment = Alignment(
            horizontal="center"
        )

        worksheet[
            celda_valor
        ].alignment = Alignment(
            horizontal="center"
        )

        return worksheet

## 25. Crear función reusable de gráfico

In [ ]:
if OPENPYXL_DISPONIBLE:

    def crear_grafico_barras(
        worksheet,
        min_row,
        max_row,
        col_categoria,
        col_valor,
        titulo
    ):
        chart = BarChart()

        chart.title = titulo

        datos = Reference(
            worksheet,
            min_col=col_valor,
            min_row=min_row,
            max_row=max_row
        )

        categorias = Reference(
            worksheet,
            min_col=col_categoria,
            min_row=min_row + 1,
            max_row=max_row
        )

        chart.add_data(
            datos,
            titles_from_data=True
        )

        chart.set_categories(
            categorias
        )

        return chart

## 26. Plantilla conceptual de informe

Un informe automatizado profesional puede seguir:

```text
Datos
  ↓
Resumen
  ↓
KPIs
  ↓
Gráficos
  ↓
Dashboard
  ↓
Metadatos
```

Lo importante es que todas estas hojas se generen de manera reproducible.

## 27. Mini-práctica integrada

Construye un informe Excel completo a partir de un DataFrame con al menos 500 filas.

Debe incluir:

1. hoja `Datos`;
2. tabla Excel;
3. formato condicional;
4. hoja `Resumen`;
5. agregaciones;
6. gráfico de barras;
7. gráfico circular;
8. gráfico de líneas;
9. hoja `Dashboard`;
10. mínimo 4 KPIs;
11. título;
12. gridlines ocultas;
13. formatos numéricos;
14. estilos consistentes;
15. hoja `Metadatos`;
16. guardado;
17. reapertura;
18. validación de hojas, tabla y gráficos.

## 28. Checklist de informe profesional

- [ ] ¿Existe una hoja de datos?
- [ ] ¿Los datos están tabulados?
- [ ] ¿Existe una tabla de Excel?
- [ ] ¿Hay formatos numéricos?
- [ ] ¿Hay formato condicional cuando aporta valor?
- [ ] ¿Existe resumen?
- [ ] ¿Hay KPIs claros?
- [ ] ¿Los gráficos tienen título?
- [ ] ¿Los gráficos usan datos correctos?
- [ ] ¿Existe dashboard?
- [ ] ¿Los anchos son adecuados?
- [ ] ¿La estética es consistente?
- [ ] ¿Existe hoja de metadatos?
- [ ] ¿Se ha guardado?
- [ ] ¿Se ha reabierto?
- [ ] ¿Se han validado los elementos esperados?

## 29. Ejercicios finales

### Ejercicio 1 — Estilos
Crea una función reusable para cabeceras.

### Ejercicio 2 — Tabla
Convierte un rango en tabla Excel.

### Ejercicio 3 — Condicional
Resalta valores superiores a un umbral.

### Ejercicio 4 — Color scale
Aplica escala de tres colores.

### Ejercicio 5 — Barras
Crea gráfico de barras.

### Ejercicio 6 — Circular
Crea gráfico circular.

### Ejercicio 7 — Líneas
Crea gráfico de evolución temporal.

### Ejercicio 8 — Dashboard
Crea una hoja específica.

### Ejercicio 9 — KPI
Construye cuatro tarjetas.

### Ejercicio 10 — Resumen
Crea tabla por región.

### Ejercicio 11 — Metadatos
Añade una hoja de trazabilidad.

### Ejercicio 12 — Visual inconsistency
Provoca un ancho absurdo y corrígelo.

### Ejercicio 13 — Gridlines
Ocúltalas en el dashboard.

### Ejercicio 14 — Reutilización
Crea funciones para gráficos y KPIs.

### Ejercicio 15 — Validación
Comprueba número de gráficos.

### Ejercicio 16 — Plantilla
Diseña la estructura estándar de tus informes.

### Ejercicio 17 — Reapertura
Valida el workbook final.

### Ejercicio 18 — Reto
Genera automáticamente un informe Excel profesional listo para entregar.

## 30. Preguntas de reflexión

1. ¿Qué diferencia hay entre datos e informe?
2. ¿Qué aporta una tabla de Excel?
3. ¿Cuándo usar formato condicional?
4. ¿Qué gráfico elegirías para comparar categorías?
5. ¿Y para evolución temporal?
6. ¿Qué riesgo tiene abusar de gráficos?
7. ¿Qué debe destacar un dashboard?
8. ¿Qué convierte un KPI en útil?
9. ¿Por qué automatizar estilos?
10. ¿Qué aporta una hoja de metadatos?
11. ¿Por qué validar gráficos y tablas?
12. ¿Qué partes deberían convertirse en funciones reutilizables?
13. ¿Cómo usarías este notebook como plantilla para múltiples clientes?
14. ¿Cómo conecta esto con generación masiva de informes?

## 31. Cierre

Ya sabemos construir un Excel funcional y visualmente profesional.

El siguiente laboratorio avanzará hacia:

```text
plantillas
   ↓
parametrización
   ↓
múltiples clientes
   ↓
múltiples archivos
   ↓
generación masiva
   ↓
informes automáticos
```